In [1]:
from astroquery.gaia import Gaia
import matplotlib.pyplot as plt
from os.path import isfile
from astropy.table import Table

# Plot-Formatierung
plt.rcParams['font.size'] = 24.0
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelsize'] = 'medium'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['lines.linewidth'] = 2.0

### Skip the following if you dont want to query the database yourself

In [2]:
Gaia.login() # You might want to create your own account if you want to query the data yourself

INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


#### Printing Information about available tables and table contents

In [3]:
def print_available_gaiadr3_tables():    
    tables = Gaia.load_tables(only_names=True)
    for table in tables:
        name: str = table.get_qualified_name()
        if name.startswith("gaiadr3"):
            print(name)
print_available_gaiadr3_tables()

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
gaiadr3.gaiadr3.gaia_source
gaiadr3.gaiadr3.gaia_source_lite
gaiadr3.gaiadr3.astrophysical_parameters
gaiadr3.gaiadr3.astrophysical_parameters_supp
gaiadr3.gaiadr3.oa_neuron_information
gaiadr3.gaiadr3.oa_neuron_xp_spectra
gaiadr3.gaiadr3.total_galactic_extinction_map
gaiadr3.gaiadr3.total_galactic_extinction_map_opt
gaiadr3.gaiadr3.commanded_scan_law
gaiadr3.gaiadr3.allwise_best_neighbour
gaiadr3.gaiadr3.allwise_neighbourhood
gaiadr3.gaiadr3.apassdr9_best_neighbour
gaiadr3.gaiadr3.apassdr9_join
gaiadr3.gaiadr3.apassdr9_neighbourhood
gaiadr3.gaiadr3.dr2_neighbourhood
gaiadr3.gaiadr3.gsc23_best_neighbour
gaiadr3.gaiadr3.gsc23_join
gaiadr3.gaiadr3.gsc23_neighbourhood
gaiadr3.gaiadr3.hipparcos2_best_neighbour
gaiadr3.gaiadr3.hipparcos2_neighbourhood
gaiadr3.gaiadr3.panstarrs1_best_neighbour
gaiadr3.gaiadr3.panstarrs1_join
gaiadr3.gaiadr3.pansta

In [4]:
def print_columns(table = "gaiadr3.gaia_source"):
  gaiadr3_table = Gaia.load_table(table)
  print(f"{'NAME':<35}{'UNIT':<20}{'DESCRIPTION':<200}")
  for column in gaiadr3_table.columns:
    print(f"{str(column.name):<35}{str(column.unit):<20}{str(column.description):<200}")
print_columns()

NAME                               UNIT                DESCRIPTION                                                                                                                                                                                             
solution_id                        None                Solution Identifier                                                                                                                                                                                     
designation                        None                Unique source designation (unique across all Data Releases)                                                                                                                                             
source_id                          None                Unique source identifier (unique within a particular Data Release)                                                                                                               

### Downloading Gaia Data

In [5]:
# TAP-query for downloading data
RERUN = False
filename = "C:/Users/Ruben/Desktop/40pc_white_dwarfs.vot"
if (not isfile(filename)) or RERUN:
    query = """
    SELECT 
        l, 
        b, 
        ra,
        ra_error,
        dec,
        dec_error,
        parallax, 
        parallax_error,
        pmra, 
        pmra_error,
        pmdec, 
        pmdec_error,
        phot_g_mean_mag, 
        teff_gspphot,
        bp_rp,
        radial_velocity,
        radial_velocity_error
    FROM 
        gaiadr3.gaia_source AS gdr3
    WHERE 
        source_id IN (4994877094997259264,431635455820288128,2312821266302222720,2414888694102153088,537563127588521600,2766234439302571904,2875903332533220992,383108338321272448,384636109728592768,395234439752169344,420531621029108608,2541549062071571968,2545505281002947200,4996458845552956928,4689789625044431616,2418116963320446720,2309122753616241280,2855386170682263424,2855790657816672000,385105360675267840,529594417061837824,4920057871348614272,2747384888699406080,2555215995900584448,4920160332088386304,4703464251158954496,2553935752048977792,2528728726428345216,2807462655009880320,2364319061964016512,2544456862306151680,4993117773313626496,4906743988126708992,2319052851148048256,2522401586766106624,418491412783587200,2349916559152267008,4926464691244602752,2377863773908424448,2377344185944929152,2523840155996531456,5006232026455470848,2552928187080872832,2356519298275043200,4925740628477887232,4987578158855872384,2803596046661176576,2582335342824976768,2472557872820632320,426122397136335872,2524879812959998592,2552121179905893888,2531326283993100416,2790494815860044544,2790494850219788160,307323228064848512,4687445500635789184,2533575369771883648,2591754107321120896,2456160271800004096,320029150076023808,4983839647522981504,372111985092019840,4934162784467241472,5037084872486444928,2587268134239449728,406775841506041344,2585189473147457408,5016473702391268096,396963005168870528,398672715686799488,4912620293662031104,2457759374023232768,2484544095751034496,2480523216087975040,5139880551029408768,4617960488907213184,2589304876450784128,5012346101380568832,572487740053132288,2568588664341691520,4698424845771339520,303637562009656704,98092934167683072,5025127443016406144,291057843317534464,318528007466920192,4686800224727589248,518201792978858880,356922880493142016,297470774951165568,105240786245136256,78629306318257536,2574620550768898432,78649033103283584,5149836834977282048,2490975272405858048,92597914738232448,5020119579868434944,330661599315957504,104648944047154432,333010327952701696,522115156720215040,2486388560866377856,4970215770740383616,561231932144411008,332820971434386432,351429930856782464,458558784733311232,4963617807621683712,355669578975070976,455517329408362752,2516606022320239104,457474219590579328,87648226538760064,5068532996689788544,515289392829738624,459237630076876672,85787470611883008,2516322146457318144,19693180966870656,131188715200383872,127366331745660288,5146358426863612160,88996326578456192,2488960249844340352,5125186299678747008,457817164137702784,5064259336725948672,2502097283492466560,452244667407804800,81606375784491520,5070533180139377792,126342377182577408,133116227803380224,25405350031335296,4725208704209795712,4725152392894633984,5128633195616949120,2495751967528809216,453562088496088320,139623068897753856,20484382662003968,5077717389114829056,4640029027306103296,143076256963396480,4696032750850438272,439494077735062144,4645960583300470656,5065010887285008128,8578256576520320,4618550411255512448,6963383233077632,4626680917489564160,466384799259137920,5179920984941752448,439905192004402304,4646535078125821568,492442090962517376,5167424240722533760,5181816233750415488,4672306015773211008,4733604373137759616,4850844228558819584,5167510797199003904,235842052999634944,4620280217923548800,5058635403471767680,4613612951211823104,4613612951211823616,239721228805415296,234469931207274112,62884540326096896,3261591090771914240,54253618862057728,4723118914857785472,5060587895604134400,5053839127592396032,3264551560189562112,5088251195844051328,4860061541910268160,63126196662620416,4829340465475546880,44901791432527232,3249479592234269056,3249479592235301376,63054590968017408,66837563803594880,3270079526697712768,243645699341060608,3251244858154433536,4667446586695129472,3302846072717868416,4683172420470864256,244214799689691904,53278867446391040,3301319572621418368,250862824946594816,39387328302768640,66144699680456320,46896958359881728,48829075167625472,3189621320226676736,551153263105246208,5090109228757394048,3195919254111315712,4678664766393827328,229143725086190336,4782553840532147840,3191738120628213760,4885878590326674560,232990572675079296,470639806179201792,4789516154317811456,3307009304776119936,256832412872262656,3283853143218651264,4839901583898106496,3198881613315343872,3202808828330265088,470826482637310848,2978374522004181888,4864752883148064512,4867574023826934272,3282093301842469376,4891567154251463680,159277625222514048,4677461862018523776,151650935831913216,3186021141200137472,3292685210887016192,3173236054352419072,4815192671404177280,4893995356962398208,199060743352070528,201881364339401728,3228023859770327424,4764692171059541120,2976789094955826560,2982808337003815040,283928743068277376,3392181976588782336,2986577153625081344,2986473765172532864,3242153305741855744,261664427174056320,3422405214775411840,4652123895783242112,3209308900555979776,208021900557436800,4805691447831511680,195470288131984128,263082591016645504,190802650815160960,4769136809374612992,194394347281717504,481698110012697728,197048460976343168,190998058945380608,2967199704297025664,3455921181049073280,3429296884940000000,3337021260634868224,3346603611845335424,2914000658819492864,3218697767783768320,3011223668834627328,3022956969731332096,198027266851070848,3346787883122375680,3320184202856435840,4650712810002383104,4767805506952137600,3024247796382652800,3348678699526809600,3329569015639064192,1114054838014073984,5482551252566796928,5210778263380292096,1007682723024253184,960039814744267520,3324181683539044224,5573532025833121408,2940238304793401472,3117320802840630400,3382277296672027136,995112350178946048,2895487456076198656,957295438016687616,3383000470383852672,3103811515783541504,3326650224581677312,3386162214851684864,2947050466531873024,2925551818747071488,3126453655659835136,943770757800160384,2926944659464115328,1115546944012493696,3358418684623972864,1100655330324351616,5564029702750970112,5564028981196462336,1100267237077449728,5564171814627287296,3129655296079487872,3112162508466471808,3160815489969826944,3052418589963163264,890661253803216896,3052844272764398208,5480556532316697216,1140292483988569856,883006556929519872,980918204822332416,3051506991743533184,5504790696302830208,5280944182024023552,946030529073021440,3059515898856688000,5589354543620145024,3156974449176326528,974375354722176768,885100916128136064,5605383430285597696,3169486960220617088,3032020450247993600,3166841329084630784,983979721933760768,5510964144860346496,1089400763661597440,5536077746353130240,976040702520790400,5717278911884258176,874900643675606912,5588614164276695424,926002203218885504,3151407827963355136,3144837318276010624,3144837112115126400,3080914632816248448,1084764020047940096,5273943488410008832,925532265076649472,984190381489031424,5513896164414899456,5602379190880667904,5698587862747531008,5725503293214289280,1081813343155426560,922434906461782272,923023248261420672,933287876501371520,5597759970724418688,680099824985004288,3145802036649641472,908962109449446656,5274517467840296832,5320436784269889792,905923368548465152,876571935709366272,5290126272348542080,5544743925212648320,657056745624156416,675615677264385536,931573222477949696,5516345223493485952,676031052142751488,5271072526109138176,5548080118369905408,5723025990435533056,5272690766709543680,5753337533146097280,5322552760038496512,710766750855439360,710947968027922176,5709868512742196608,5748802661860766208,5322090003089341440,5639391810273308416,5734737438536674432,690287663506579328,1042071701528617856,1042071701528617728,5625513014289760128,717393648787722624,716504796716020352,3073747187092773632,5302618648583292800,611645983387812992,5652718097353105664,5652689406971618048,5755957119598921728,1117334024067935872,912718071240545152,5764485618978306176,712888090655562624,715357559411376128,684598618544000384,5620763437599189376,5762406957886626816,711744456827031680,1123700235048742016,5648566371510973824,5624029566946316928,611074413433751680,5651964996310470144,688606063549945600,5649808720867457664,636732410620758016,1040160612880069888,1022780838739029120,5427528254746168192,591040864898749312,5297051821220130560,818602457173176192,5423545067716723712,5423896396040719744,3843957354387777152,700531568527365376,5314177402013456256,1039163458912998272,633911196927706880,5219215228420645248,5681903877597244032,5633102260158519936,696261653777188864,5436014972680358784,5436014972680358272,5432789383518999168,826275295988399232,5688717241916173568,5425208663166556288,5410698683096655360,3820383996887145088,820105214691139584,639665392247496576,643183348419998336,1020653077580086784,616396182856185728,1066726497434084864,5242316444456199808,642685200933153408,615733593956317568,808030648576069760,3828424828500179584,3822028007288795264,1053211437944753024,5407613384750351232,5473389537569200896,740483560857296768,5669427512997660800,5356710428806242816,805470233890349440,3875651975353757440,3875652014008894720,5446665014103742336,5446784345474819968,1146403741412820864,833470465721350784,755877620910173696,1052520154368111872,5441714531716273920,3882611201058534400,3855631797052747136,3883918657822146944,5469171261208778240,3750749378584132992,3754712881779364992,5233071514480420864,1076941716370493696,5367774809996936960,3553682127126319360,3870354528331257984,3884899559633556224,5231870985222281088,3554395813252626048,5391794195555570048,1055313016981775488,776762672481353984,3557394009663078912,730459763934332416,5353122722363361920,3549471753507182592,3763445409285757824,3789156870225942656,5456685104084844032,3982007636324256000,777395029106166272,776981269136616960,3552845261339955200,3968635204109066880,861050512312844672,5401688425816913920,3788194488314248832,3818473629793533312,843807902246527232,3990494251184010496,3810099989754827136,3817262208497857024,5376644127919488896,3790040465258127616,5374565879145559424,3810933247769901696,3783206210217512320,3978879594463300992,3978862277154958592,859082970614616448,3482983495102507904,5398247534240054528,3478127467639543296,859567752163281792,4019458647338779648,3585097235918075776,3464893058489831552,3793132257595347200,5332606522595645952,863131372427958912,4021565827014024576,3794567429507510528,5224999346778496128,5377849123945711872,5381346739148118016,3480776843983381632,4004185576130620288,3490527755479959936,1058284412796260480,3487220772397809536,3479615106870788864,4004395720290994048,5334619419176460928,3920187251456610816,3926968661219149184,3795052348495488896,3891115064506627840,5341271911184522880,1537794524729363712,3575728709655386752,3575770010060921728,3489719481290397696,3904628406009296896,1575357587146077056,1717341818608187648,3694399755554510720,6151294355090597504,1683330453627242752,6054148143441683072,3905335598144227200,3947104533054775168,1518943638389520384,5860131207828395648,4015547856277853824,3583181371265430656,3696778892558087552,3583402849843287680,6127190796769848960,3708578473389974528,6133033635916500608,1534384148897669248,6127333286605955072,1541286711100812160,3935942939548822272,3500086050578451712,3528871819044810368,6158704208764041472,5784295623056963200,6079710620510474240,3530520910392199680,1570514066627694336,1517239773324267904,1531097433767946240,1460689760003983232,3704392873141718912,3690395231125833600,1679365202380970880,1459546263999675264,6154946657841163392,3629758603668233728,1686322048672412288,5784706428090844160,1726678630833373824,6085402414245451520,1566603962760532736,3691685882071367936,3506567328028533120,6067215083178616704,1556005701461744640,3623233040812235904,3943650619138622848,1688618481786030336,3506061587037686144,5787859896160384384,6063252374582712320,1551062778220116992,5869567658943170048,6188345358621778816,3630035787972473600,1468213546275054208,6087659745978472064,5845312191917620224,1552488776081383040,1472029470098019072,3713594960831605760,5850533227210203520,1686708050268594944,3714914271705535360,6165095738576250624,6165010320266420096,5851860818806598528,1500607765872799616,1658578797618935680,3725570772761744384,1251824057289839744,3728074738695246336,1451566149955227520,1502063317405058304,6177238676273826304,3727155340815968256,3618657732410663808,6178524211524592640,6275184065428686464,6114344102909485824,1505825635741455872,1671668067321674368,5846206030463663232,1454347089739329152,3643555726544985088,1498447607777405312,3667514634669861120,5790182751914903424,5867776696271127424,1232045934759720192,6296317052576778240,6118079453145005312,5852538324131652736,6329136310728635776,6093257119157372160,5898935893701856128,1176717792385803136,1507571286545131648,6272326022391660928,6271903947364173056,5772718006135360128,1493367245581725184,5893318763718868736,5785654859950570496,6310804634396281984,1282448170543051520,1282448170543051648,5799644049485006848,6282457918962299776,1281989124439286912,6227687980608264064,6332763530870415488,1620030637207462912,1276688069644366592,6255777749623059584,5903884280152869632,1722236328978172928,1645204475617697536,5902612969841664768,6206195620664214400,1602197696772907648,1273685372108354176,5826604589994061312,6265877455415860224,5827454649955072256,4424031479858305408,1401010605309570816,1193520666521113344,1641386833807056640,1370654257498865536,5985749857190372480,1218051664291152000,1376326912864142336,6009537829925128064,1598771000065137536,6008386881767536128,6008581907636919936,1202826348825240832,1403348682426861568,4348098485293072128,4425632987265111680,6264127170346899712,6250213984568447872,1404831472640252928,1372458109403442432,5998095590373665792,1622697949337226112,4349513797276615680,4341773063622911872,4411572123331402880,5806672265237931776,1647162396588999552,4341495230772911616,1323922779935068160,4458207634145130368,1385719147346936064,4452997701376633728,6246049446837287680,4355229123137665792,5931881969426463616,6018034958869558912,6022366686772364288,1201036206454896000,4452521234885949184,4323956302321933952,1331106782752978688,1653044367185115264,6018613473779340928,6044265144466741888,4440264291578812928,6018693257096471424,1703379704562897280,1431176943768691328,4466388790929771904,1405343643196929536,5765270154886903168,4435778215414219520,1326398777041821568,1425909733315616000,4462612140287443840,5929529014509678720,1431783457574556672,4126670518631322880,6027138331058857088,4379328051494006784,4334641562477923712,1631578537252535040,1351956512512484480,4125468645047515904,1306197930941817984,4565048312887877888,5775307733975564160,4340322876499078400,1358325021299899136,1313405848136604672,1358301480583401728,5765288880944438144,5808208420421074560,4444590625015876864,5923906524348891904,4380188694219651456,5938773880045035776,4139348334376604928,4379812558164849408,1408135749896104192,1408135749896103936,4573071139998034048,4108828945319007744,4388138816124225792,4139531467491239680,1638563322306634368,4361621688038664064,4136103572502555264,1341543072245722752,4359722208685335552,5915797694789556096,4387171623850187648,4367353266060378624,1336988963803208192,4491748511228743808,5975317695158795776,4598266758185956864,4542785981266940416,1706631093589103872,4598775557191664384,4110161965722210432,1348795004265872000,5921433963191695872,4055353888055627008,4168312459956062208,4053455379420643584,4581383928942599296,4118171568655203328,5909078751020702080,1349256249394224128,4549622027311531904,4500646618315862144,5961193055261256320,4117081643422165120,4149820323678136192,5909739660590724224,4150020774057837440,4596322473734130304,5920900901901635968,5945252202546434432,1638979384378696704,1350517492310907392,4068499305485306240,1711005951573009792,5767614206300408576,4604247070649603584,4067477554248982528,4611543459874840832,4499839473701254400,4611204329256349312,5911160263973498496,2123432247756223488,6363592840482769792,4037085334973293824,6414612172876713472,4470233817461336704,6431530976766770560,6725656144031366144,4590489981163833984,6345857271249415424,4153937891610652928,2149331587745863680,4578913738632417920,4497414466452138496,4146666271458052352,4094555467661923328,4523585076572785408,4585067258532443776,2158285185808357504,2149253075743572224,6635520414133666944,6437614815119680768,4154063678315488640,4528439381757452928,4152557420406043264,4484277256704949376,4484289866726156160,4484184592790777728,4483974792231866624,4257461275049675008,4257063458004688896,4257063453704172416,2150504594853811456,2118649750133781376,6432020637402543616,4589139574728058880,4525569007873380736,2256410856215182464,6709854989379725056,6651133479247436032,6631915390382416256,4107012041007171456,4280632829779587072,6730516225919672704,6705845452725619072,4512265810525783680,4539227892919675648,4504577986571873280,4252064631569619712,4203108841963846016,6761581964903088256,2292861388958880640,2146645790077864704,4073522222505044224,4518917168694695168,2092134443120038528,2146576589564898688,4088653838978654336,2262849634963004416,4254454797960984192,6658005048964693760,6718079679950008704,6763036202147206912,4268167357267580160,2140481412496465152,2039140284770609152,4320094439580536320,4320303621677848832,2295446546953958272,6664729284121316864,2052891361294411520,2127093140445053696,6742379406616165376,4201781696994073472,4293873732939569920,4319908862597055232,4288942973032203904,2018864362679341824,2127566548919332608,2142307563871222912,4213409341688406912,4314903198513595776,4215241712185612544,6744843579679059968,2024985481361040384,2025389380082340992,2248748668919802496,2301882675705225472,6740493675455190784,6671045050707117568,2080526555267049984,2080526555267050496,4181798519823760256,4298029268399256704,4240231824768647040,2078430778727685632,2073772770741915264,6368021054841781376,4235280071072332672,6666783962114636288,6670827416123874688,4237555506083389568,4190813690536580608,6673731810450381056,6865904860773722496,2237893023118101504,6424223313252632576,6874124023727679104,6853784501721502720,6749419923164242816,6853523539508720000,2053953008490747392,6443365570172526848,6672244029484054272,4249667902270614272,4230380819051252736,6361559602963567744,6692573110423893376,6879524790480960896,6693352488073255936,6429048245152936320,6797171060323993728,6680097978480311040,2185261016407220224,2302010356492847744,6468965052723781888,6679105566157898880,4217793816094052480,6429465338016396928,6875432476922523520,2064284054100290048,6862687522250677376,2064689567732385792,1831553382794173824,6683212727417918848,6857939315643803776,6681773947733560192,6424566979354709248,2169971345155578752,1871118140493076224,6470278694244646912,1844125748497557632,6454251250683840896,6805808514433280000,6805792571514600960,6808651507904773888,6857295585945072128,6913810483611035776,2188860027203347968,2169025009235266816,6478328218869704192,1864760695541016832,1789361097242243584,1841254644460354688,6479860113444922368,6774018369099513216,2690697646876721152,1737167215848315264,6912866381081015552,6831993452567326592,6788656957673130112,6348672845649310464,1739921801713625600,1841683423932168832,2176116580055936512,6462911897617050240,2191146977029443584,6791196382856581376,2685959542034846464,6583325635088476544,2693095097621419648,6578917727331681536,2177776331525931136,2274076297221555968,2300234782654298624,1797472370615364992,1951870157081161216,1977417206686680704,6842831888437047680,6585792870460638336,6578616598584224640,6844375121726139520,6589369272547881856,6592315723192176896,1792830060723673472,2701893698904233216,2667464656943675392,6584418167391671808,2696628687474414208,1800298527816525824,2669936427801840256,2202703050401536000,2693940725141960192,2679976510857026048,6617996741403360128,6558209044297239424,1892992267183979776,2676567307551465088,2676566272464334720,6412251624488840448,2199371701965748992,2683452758602312192,6358158435541361792,6613289285448236288,6397887600292497408,6409446323650635264,1900382604528495744,2735175263041913088,2198431172852758656,2600033326799287296,1955134710179436672,6573778541262656256,1907041590544054656,2006217676803960960,1999615350008375552,6404417771644764160,2730707260103011712,2205493129867600256,2730508416002618752,1875613386395668864,6357629089412187648,6357630601240673792,2736054627915080448,1875301369907249024,6505318682415051520,2733055335904034432,6505113009316709760,2628943473222829440,6520844168856463104,6506903598362207872,2731866347221858432,2836609093355562496,1884744525522874880,1929287700069881856,2410908771246898176,2286958798223194624,2611561706216413696,2610488514148351360,2711324446359728384,1989342372349280384,2208530698238308736,6554977369168846720,1936315366080098432,6552878165248320896,1929838143078434432,2842462137347560320,2386223463892847872,1996725077535283200,6387649708219253248,2812250821990695936,2631876970245863552,2638553754605793408,2407167579853954688,2818957013992481280,6527675297155337472,2631967439437024384,2405805697263561600,2869130517001766400,2813020961166816512,2814629409239942272,2660358032257156736,1923682286712356992,6389551313580046464,2865535629374939520,2395444208921491456,2439184705619919488,2393875961742886656,2826254713186397440,6485572518732377856,2394366515727615104,2871730307948650368,2763719512612656000,2742789930821144320,6531195177474061824,6521660556236440960,2867032958053059200,2448933731627261824,2313836325604479616,6350786278796634112,6521875098442800512,1921351390779081600)
    """
    #AND parallax_over_error > 5 -- Ensures useful parallax measurement
    #teff_gspphot BETWEEN 4000 AND 5000

    # Perform TAP query
    job = Gaia.launch_job_async(query)  # This runs for <30min
    results = job.get_results()

    # Print the first few rows
    # print(results)

    # Save to file
    results.write(filename, format="votable", overwrite=True)
    print("Data download complete.")
else:
    print("Data file found. Skipping Gaia query.")

INFO: Query finished. [astroquery.utils.tap.core]
Data download complete.


### mit T und M sollten nur K-dwarfs übrig bleiben vgl. Hertzsprung-Russel

### End of Gaia query part

#### Loading Data-Table

In [7]:
# tab = Table.read("C:/Users/Ruben/Desktop/All_candidates_200pc.vot")
# print(len(tab)) #632951
tab = Table.read("C:/Users/Ruben/Desktop/All_candidates_200pc_noparallaxovererror.vot")
# print(len(tab)) #644675